# 01. Transfer Learning

## 학습 목표
- Pre-training → Fine-tuning 패러다임이 왜 작동하는지 이해
- Feature Extraction vs Fine-tuning 전략 비교 실험
- Layer-wise Learning Rate Decay 구현
- Unfreezing 전략에 따른 성능 차이 이해

## 참고 자료
- [ULMFiT 논문 (Howard & Ruder, 2018)](https://arxiv.org/abs/1801.06146)
- [BERT 논문 (Devlin et al., 2019)](https://arxiv.org/abs/1810.04805)

---

In [ ]:
# Google Colab 환경 설정
!pip install -q transformers datasets accelerate scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Pre-training → Fine-tuning 패러다임: 왜 작동하는가

### 핵심 아이디어

NLP에서 Transfer Learning의 핵심 흐름:

```
Pre-training (대규모 텍스트)          Fine-tuning (작은 태스크 데이터)
━━━━━━━━━━━━━━━━━━━━━━━━━    →    ━━━━━━━━━━━━━━━━━━━━━━━━━
목적: 언어 이해 능력 학습            목적: 특정 태스크에 적응
데이터: 수십~수백 GB 텍스트         데이터: 수천~수만 개 레이블
시간: 수 주 (GPU 수백 개)            시간: 수 시간 (GPU 1개)
비용: 수억 원                        비용: 수만 원
```

### 왜 작동하는가?

1. **Lower layers**: 범용 언어 패턴 학습 (문법, 구문, 기본 의미)
2. **Middle layers**: 좀 더 추상적인 언어 표현
3. **Upper layers**: 태스크에 특화된 표현

사전학습 모델은 이미 **범용적인 언어 지식**을 보유하고 있으므로, 적은 데이터로도 특정 태스크에 빠르게 적응할 수 있다.

| 접근 방식 | 설명 | 장점 | 단점 |
|-----------|------|------|------|
| From Scratch | 랜덤 초기화부터 학습 | 완전한 제어 | 대규모 데이터 필요 |
| Feature Extraction | 사전학습 가중치 동결 | 빠름, 적은 데이터 OK | 표현력 제한 |
| Fine-tuning | 전체 가중치 업데이트 | 최고 성능 | 과적합 위험, 더 많은 계산 |

---
## 2. Feature Extraction: 사전학습 모델의 가중치 동결

사전학습 모델을 **고정된 특성 추출기(feature extractor)**로 사용하는 방식.

$$\text{output} = \text{Classifier}(\text{BERT}_{\text{frozen}}(\text{input}))$$

BERT의 모든 가중치를 동결(`requires_grad=False`)하고, 마지막에 추가한 분류 레이어만 학습한다.

In [ ]:
# 데이터 준비: IMDB 영화 리뷰 감정 분류
dataset = load_dataset('imdb')

# 학습 시간을 위해 서브샘플링
train_dataset = dataset['train'].shuffle(seed=42).select(range(2000))
test_dataset = dataset['test'].shuffle(seed=42).select(range(500))

print(f"Train 샘플 수: {len(train_dataset)}")
print(f"Test 샘플 수: {len(test_dataset)}")
print(f"\n예시:")
print(f"Text: {train_dataset[0]['text'][:200]}...")
print(f"Label: {train_dataset[0]['label']} (0=negative, 1=positive)")

In [ ]:
# 토크나이저 + 전처리
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# PyTorch 포맷으로 변환
train_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"토큰화 완료! input_ids shape: {train_tokenized[0]['input_ids'].shape}")

In [ ]:
class FeatureExtractionClassifier(nn.Module):
    """Feature Extraction 방식: BERT 가중치 동결, 분류 헤드만 학습"""
    
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        
        # 핵심: BERT의 모든 파라미터 동결
        for param in self.bert.parameters():
            param.requires_grad = False
        
        # 학습 가능한 분류 헤드만 추가
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )
    
    def forward(self, input_ids, attention_mask):
        # BERT 출력 (gradient 계산 불필요)
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # [CLS] 토큰의 표현을 분류에 사용
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

# 모델 생성 + 학습 가능 파라미터 확인
fe_model = FeatureExtractionClassifier(model_name).to(device)

total_params = sum(p.numel() for p in fe_model.parameters())
trainable_params = sum(p.numel() for p in fe_model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}")
print(f"학습 가능 파라미터: {trainable_params:,}")
print(f"학습 비율: {trainable_params/total_params*100:.2f}%")
print(f"→ BERT 가중치는 동결, 분류 헤드만 학습!")

---
## 3. Fine-tuning: 전체 가중치 업데이트

BERT의 모든 레이어를 포함하여 전체 모델의 가중치를 업데이트하는 방식.

$$\theta_{\text{new}} = \theta_{\text{pretrained}} - \eta \nabla_{\theta} \mathcal{L}(\theta)$$

- $\theta_{\text{pretrained}}$: 사전학습된 가중치 (좋은 초기값)
- $\eta$: 학습률 (보통 사전학습보다 훨씬 작은 값 사용: $2 \times 10^{-5}$)
- $\mathcal{L}$: 태스크 손실 함수

In [ ]:
class FineTuningClassifier(nn.Module):
    """Fine-tuning 방식: BERT 전체 + 분류 헤드 함께 학습"""
    
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        
        # BERT 가중치도 학습 가능 (동결하지 않음!)
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_labels)
        )
    
    def forward(self, input_ids, attention_mask):
        # BERT 출력 (gradient 계산 포함!)
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_output)

ft_model = FineTuningClassifier(model_name).to(device)

total_params = sum(p.numel() for p in ft_model.parameters())
trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total_params:,}")
print(f"학습 가능 파라미터: {trainable_params:,}")
print(f"학습 비율: {trainable_params/total_params*100:.2f}%")
print(f"→ 모든 파라미터가 학습됨!")

---
## 4. 실험: BERT로 감정 분류 - Feature Extraction vs Fine-tuning

동일한 데이터, 동일한 에폭으로 두 방식의 성능을 비교한다.

In [ ]:
def train_model(model, train_data, test_data, epochs=3, lr=2e-5, batch_size=16, model_name_str='model'):
    """모델 학습 + 평가 루프"""
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size)
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )
    criterion = nn.CrossEntropyLoss()
    
    train_losses = []
    test_accs = []
    
    for epoch in range(epochs):
        # Train
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Evaluate
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                logits = model(input_ids, attention_mask)
                preds = torch.argmax(logits, dim=-1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(batch['label'].numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        test_accs.append(acc)
        print(f"[{model_name_str}] Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, Test Acc: {acc:.4f}")
    
    return train_losses, test_accs

In [ ]:
# Feature Extraction 학습 (높은 학습률 사용 가능 - 분류기만 학습하므로)
print("="*60)
print("Feature Extraction 학습")
print("="*60)
fe_model = FeatureExtractionClassifier(model_name).to(device)
fe_losses, fe_accs = train_model(
    fe_model, train_tokenized, test_tokenized,
    epochs=3, lr=1e-3, model_name_str='Feature Extraction'
)

In [ ]:
# Fine-tuning 학습 (낮은 학습률 필수 - 사전학습 가중치 보존)
print("="*60)
print("Fine-tuning 학습")
print("="*60)
ft_model = FineTuningClassifier(model_name).to(device)
ft_losses, ft_accs = train_model(
    ft_model, train_tokenized, test_tokenized,
    epochs=3, lr=2e-5, model_name_str='Fine-tuning'
)

In [ ]:
# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss 비교
ax = axes[0]
ax.plot(range(1, 4), fe_losses, 'o-', label='Feature Extraction', color='blue')
ax.plot(range(1, 4), ft_losses, 's-', label='Fine-tuning', color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Training Loss Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy 비교
ax = axes[1]
ax.plot(range(1, 4), fe_accs, 'o-', label='Feature Extraction', color='blue')
ax.plot(range(1, 4), ft_accs, 's-', label='Fine-tuning', color='red')
ax.set_xlabel('Epoch')
ax.set_ylabel('Test Accuracy')
ax.set_title('Test Accuracy Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Test Accuracy:")
print(f"  Feature Extraction: {fe_accs[-1]:.4f}")
print(f"  Fine-tuning:        {ft_accs[-1]:.4f}")
print(f"  차이: {ft_accs[-1] - fe_accs[-1]:.4f}")

---
## 5. Layer-wise Learning Rate Decay

ULMFiT (Howard & Ruder, 2018)에서 제안된 기법으로, **아래 레이어는 작은 학습률**, **위 레이어는 큰 학습률**을 사용한다.

직관: 아래 레이어는 이미 범용적인 언어 지식을 잘 학습했으므로 크게 바꿀 필요가 없다.

$$\eta_l = \eta_{\text{base}} \cdot \xi^{L-l}$$

- $\eta_l$: $l$번째 레이어의 학습률
- $\xi$: Decay factor (보통 0.9~0.95)
- $L$: 전체 레이어 수
- $l$: 현재 레이어 번호

In [ ]:
def get_layer_wise_lr_params(model, base_lr=2e-5, decay_factor=0.9):
    """
    Layer-wise Learning Rate Decay 적용.
    BERT의 각 Transformer 레이어에 다른 학습률을 할당한다.
    """
    param_groups = []
    
    # Embedding layer: 가장 낮은 학습률
    num_layers = model.bert.config.num_hidden_layers  # 보통 12
    
    # Embeddings
    lr = base_lr * (decay_factor ** num_layers)
    param_groups.append({
        'params': list(model.bert.embeddings.parameters()),
        'lr': lr,
        'name': 'embeddings'
    })
    
    # Transformer layers (0 ~ 11)
    for i, layer in enumerate(model.bert.encoder.layer):
        lr = base_lr * (decay_factor ** (num_layers - i - 1))
        param_groups.append({
            'params': list(layer.parameters()),
            'lr': lr,
            'name': f'layer_{i}'
        })
    
    # Classifier head: 가장 높은 학습률
    param_groups.append({
        'params': list(model.classifier.parameters()),
        'lr': base_lr,
        'name': 'classifier'
    })
    
    return param_groups

# Layer-wise LR 확인
ft_model_llrd = FineTuningClassifier(model_name).to(device)
param_groups = get_layer_wise_lr_params(ft_model_llrd, base_lr=2e-5, decay_factor=0.9)

print("Layer-wise Learning Rates:")
print("-" * 40)
for group in param_groups:
    n_params = sum(p.numel() for p in group['params'])
    print(f"{group['name']:>15s}: lr = {group['lr']:.2e}  ({n_params:>10,} params)")

In [ ]:
# Layer-wise LR 시각화
base_lr = 2e-5
decay_factor = 0.9
num_layers = 12

layers = ['embed'] + [f'layer_{i}' for i in range(num_layers)] + ['classifier']
lrs = [base_lr * (decay_factor ** num_layers)]  # embedding
for i in range(num_layers):
    lrs.append(base_lr * (decay_factor ** (num_layers - i - 1)))
lrs.append(base_lr)  # classifier

plt.figure(figsize=(10, 4))
plt.bar(range(len(layers)), [lr * 1e5 for lr in lrs], color='steelblue')
plt.xticks(range(len(layers)), layers, rotation=45, ha='right')
plt.ylabel('Learning Rate (x1e-5)')
plt.title('Layer-wise Learning Rate Decay (decay=0.9)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Embedding LR / Classifier LR = {lrs[0]/lrs[-1]:.4f}")
print(f"→ 아래 레이어는 위 레이어보다 {1/lrs[0]*lrs[-1]:.1f}배 느리게 학습")

---
## 6. 얼마나 많은 레이어를 풀어야 하는가 (Unfreezing Strategy)

### 전략들

| 전략 | 설명 | 적합한 경우 |
|------|------|-------------|
| 전체 동결 (Feature Extraction) | 마지막 레이어만 학습 | 매우 적은 데이터, 빠른 학습 |
| 상위 N개 레이어만 unfreeze | 부분적 Fine-tuning | 중간 크기 데이터 |
| 점진적 Unfreezing | 에폭마다 하나씩 unfreeze | 과적합 방지가 중요할 때 |
| 전체 Unfreeze | 모든 레이어 학습 | 충분한 데이터, 최고 성능 목표 |

### 판단 기준

- **데이터가 적을수록** → 더 많은 레이어를 동결
- **태스크가 사전학습과 유사할수록** → 더 많은 레이어를 동결
- **데이터가 충분할수록** → 더 많은 레이어를 풀어서 학습

In [ ]:
def create_partial_finetune_model(model_name, num_unfrozen_layers=4, num_labels=2):
    """
    상위 N개 레이어만 unfreeze하는 모델 생성.
    BERT-base는 12개 레이어 → num_unfrozen_layers=4이면 layer 8~11만 학습.
    """
    model = FineTuningClassifier(model_name, num_labels)
    
    # 전체 동결
    for param in model.bert.parameters():
        param.requires_grad = False
    
    # 상위 N개 레이어만 unfreeze
    num_layers = model.bert.config.num_hidden_layers
    for i in range(num_layers - num_unfrozen_layers, num_layers):
        for param in model.bert.encoder.layer[i].parameters():
            param.requires_grad = True
    
    # 분류 헤드는 항상 학습
    for param in model.classifier.parameters():
        param.requires_grad = True
    
    return model

# 다양한 unfreezing 전략 비교
print("Unfreezing 전략별 학습 가능 파라미터 수:")
print("-" * 55)

strategies = [0, 2, 4, 8, 12]  # unfrozen layer 수
for n in strategies:
    if n == 0:
        model = FeatureExtractionClassifier(model_name)
        name = "Feature Extraction (0 layers)"
    elif n == 12:
        model = FineTuningClassifier(model_name)
        name = f"Full Fine-tuning ({n} layers)"
    else:
        model = create_partial_finetune_model(model_name, n)
        name = f"Partial ({n} layers unfrozen)"
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"{name:>35s}: {trainable:>12,} / {total:>12,} ({trainable/total*100:5.1f}%)")
    del model

In [ ]:
# 시각화: Unfreezing 범위에 따른 학습 가능 파라미터 수
unfrozen_counts = list(range(0, 13))
trainable_counts = []

for n in unfrozen_counts:
    if n == 0:
        m = FeatureExtractionClassifier(model_name)
    else:
        m = create_partial_finetune_model(model_name, n)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    trainable_counts.append(trainable / 1e6)
    del m

plt.figure(figsize=(10, 4))
plt.bar(unfrozen_counts, trainable_counts, color='steelblue')
plt.xlabel('Number of Unfrozen Layers')
plt.ylabel('Trainable Parameters (M)')
plt.title('Trainable Parameters by Unfreezing Strategy')
plt.xticks(unfrozen_counts)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 점진적 Unfreezing (Gradual Unfreezing) 구현

ULMFiT에서 제안된 점진적 Unfreezing을 구현하세요.
- Epoch 1: 분류 헤드만 학습
- Epoch 2: 마지막 2개 레이어 + 분류 헤드 학습
- Epoch 3: 마지막 4개 레이어 + 분류 헤드 학습
- 각 에폭의 loss와 accuracy를 기록하세요.

In [ ]:
# TODO: 점진적 Unfreezing 구현
# Hint:
# 1. FineTuningClassifier 생성 후 전체 동결
# 2. 에폭 루프에서 unfreeze할 레이어를 점진적으로 늘리기
# 3. 각 에폭마다 optimizer를 재생성 (새로 unfreeze된 파라미터 포함)

# model = FineTuningClassifier(model_name).to(device)
# for param in model.bert.parameters():
#     param.requires_grad = False
#
# unfreeze_schedule = [0, 2, 4]  # 에폭별 unfreeze할 레이어 수
# for epoch, n_unfreeze in enumerate(unfreeze_schedule):
#     # TODO: n_unfreeze 만큼 레이어 unfreeze
#     # TODO: optimizer 재생성
#     # TODO: 1 에폭 학습 + 평가
#     pass

### 연습 2: HuggingFace Trainer API로 Fine-tuning

직접 학습 루프를 짜는 대신, HuggingFace의 `Trainer` API를 사용하여 BERT 감정 분류 Fine-tuning을 구현하세요.

In [ ]:
# TODO: HuggingFace Trainer로 Fine-tuning
# Hint:
# 1. AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# 2. TrainingArguments 설정 (output_dir, epochs, batch_size, lr 등)
# 3. compute_metrics 함수 정의 (accuracy 계산)
# 4. Trainer 생성 + trainer.train()

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return {'accuracy': accuracy_score(labels, predictions)}

# model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# training_args = TrainingArguments(...)
# trainer = Trainer(...)
# trainer.train()

---
## 핵심 정리

| 개념 | 설명 | 핵심 포인트 |
|------|------|-------------|
| Pre-training → Fine-tuning | 대규모 데이터로 범용 학습 후, 소규모 데이터로 특화 | 현대 NLP의 기본 패러다임 |
| Feature Extraction | 사전학습 가중치 동결, 분류 헤드만 학습 | 빠르지만 성능 한계 |
| Fine-tuning | 전체 가중치를 낮은 학습률로 업데이트 | 최고 성능, 과적합 주의 |
| Layer-wise LR Decay | 아래 레이어는 작은 LR, 위 레이어는 큰 LR | 범용 지식 보존 + 태스크 적응 |
| Gradual Unfreezing | 에폭마다 점진적으로 레이어를 풀어줌 | 안정적 학습, 과적합 방지 |
| Unfreezing 전략 | 데이터 양, 태스크 유사도에 따라 결정 | 데이터 적으면 많이 동결 |

**다음 노트북**: [02-full-finetuning.ipynb](02-full-finetuning.ipynb) - Instruction Fine-tuning과 HuggingFace Trainer 심화